# Preprocessing of station current data

Note: Something is off with directions after 2019-01-01 for Flinten7 station. They have been removed for now. 

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def convert_to_u_v(speed, direction):
    # Convert direction from CW (North = 0°) to CCW (East = 0°)
    direction_ccw_deg = (90 - direction) % 360
    direction_rad = np.deg2rad(direction_ccw_deg)

    # Calculate u- and v-components
    u = speed * np.cos(direction_rad)
    v = speed * np.sin(direction_rad)

    # Alternative calculation which achieves the same by swapping sin and cos
    #u = speed * np.sin(np.deg2rad(direction))
    #v = speed * np.cos(np.deg2rad(direction))
    return u,v

def uv_to_speed_direction(u, v):
    # Calculate speed and direction from u- and v-components
    speed = np.sqrt(u**2 + v**2)
    direction_deg = np.rad2deg(np.arctan2(v, u))
    direction_ccw_deg = (90 - direction_deg) % 360
    return speed, direction_ccw_deg

def calculate_depth_averaged_velocities(data): 
    # Set values to nan if they are exactly 0:
    data = data.copy()
    data.loc[data["Current direction"] == 0,"Current direction"] = np.nan
    data.loc[data["Current speed"] == 0,"Current speed"] = np.nan

    data['u'],data['v'] = convert_to_u_v(data['Current speed'],data['Current direction'])
    for idx in data.index.unique():
        # Loop over indices and calculate depth averaged velocities. 
        # Also convert to m/s
        data.loc[idx,'u_depth_averaged'] = data.loc[idx,'u'].mean()/100
        data.loc[idx,'v_depth_averaged'] = data.loc[idx,'v'].mean()/100
    # Round data
    data = data.round({'u_depth_averaged': 4, 'v_depth_averaged': 4})
    return data

def calculate_velocities(data): 
    data = data.copy()
    # Set values to nan if they are exactly 0:
    data.loc[data["Current direction"] == 0,"Current direction"] = np.nan
    data.loc[data["Current speed"] == 0,"Current speed"] = np.nan

    data['u'],data['v'] = convert_to_u_v(data['Current speed'],data['Current direction'])
    data['u'] = data['u']/100
    data['v'] = data['v']/100

    # Round data
    data = data.round({'u': 4, 'v': 4})
    return data

In [ ]:
swedish_to_english = { 'Strömriktning':"Current direction", 'Kvalitet':"Quality_d",'Mätdjup (m)':"Measuring depth (m)_d",
                            'Strömhastighet':"Current speed", 'Kvalitet.1':"Quality_s",'Mätdjup (m).1':"Measuring depth (m)_s"}

## Flinten7 station

In [ ]:
Flinten7 = pd.read_csv("raw_data/smhi-opendata_2_3_35137_20260520_111930.csv",index_col=0,sep=";", parse_dates=True)
Flinten7.rename(columns = swedish_to_english,inplace=True)
Flinten7.index.names = ['datetime_UTC']

# Filter data after time steps and quality flags.
Flinten7 = Flinten7.loc[(Flinten7.index >= "2023-01-01") & (Flinten7.index < "2026-01-01") & (Flinten7["Quality_d"] == "G") & (Flinten7["Quality_s"] == "G") ].drop(columns=["Quality_d","Quality_s","Unnamed: 7","Tidsutsnitt"])
Flinten7


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
Flinten7["Current speed"].plot(style='.')
plt.show()
Flinten7["Current direction"].plot(style='.')
plt.show()

### Depth averaged

In [ ]:
# Convert Current speed and directions to u and v components and calculate depth averaged velocities
Flinten7_depthave = calculate_depth_averaged_velocities(Flinten7).dropna()
Flinten7_depthave


In [ ]:
Flinten7_depthave.drop_duplicates(subset=["u_depth_averaged","v_depth_averaged"],inplace=True)
Flinten7_depthave

In [ ]:
df = Flinten7_depthave[["u_depth_averaged","v_depth_averaged"]].rename(columns={"u_depth_averaged":"u","v_depth_averaged":"v"})

df.to_csv("../observations/Flinten7_u_v_depth_averaged.csv", date_format='%Y-%m-%dT%H:%M:%S')

### All levels

In [ ]:
# Convert Current speed and directions to u and v components and calculate depth averaged velocities
Flinten7_depths = calculate_velocities(Flinten7).dropna()
Flinten7_depths

In [ ]:
df = Flinten7_depths[["Measuring depth (m)_d","u","v"]].rename(columns={"Measuring depth (m)_d":"depth"})

In [ ]:
# Convert depths to z values (zero at surface, negative values going down)
df['z'] = -df['depth']
df


In [ ]:
df.drop(columns=["depth"]).to_csv("../observations/Flinten7_u_v.csv", date_format='%Y-%m-%dT%H:%M:%S')

## Check other stations as well

In [ ]:
# LandskronaHamn
dat = pd.read_csv("raw_data/smhi-opendata_2_3_35242_20260520_111919.csv",index_col=0,sep=";", parse_dates=True)
dat.rename(columns = swedish_to_english,inplace=True)
dat.index.names = ['datetime_UTC']

# Filter data after time steps and quality flags.
dat = dat.loc[(dat.index >= "2023-01-01") & (dat.index < "2026-01-01") & (dat["Quality_d"] == "G") & (dat["Quality_s"] == "G") ].drop(columns=["Quality_d","Quality_s","Unnamed: 7","Tidsutsnitt"])
dat


In [ ]:
dat["Current speed"].plot(style='.')
plt.show()
dat["Current direction"].plot(style='.')
plt.show()

## Other two stations

In [ ]:
dirs = ["smhi-opendata_2_3_35136_20260520_111943.csv"]
stations = ["Flinten16"]

for i,d in enumerate(dirs):
    data = pd.read_csv(f"raw_data/{d}",index_col=0,sep=";", parse_dates=True)
    data.rename(columns = swedish_to_english,inplace=True)
    data.index.names = ['datetime_UTC']

    # Filter data after time steps and quality flags.
    data = data.loc[(data.index >= "2023-01-01") & (data.index<"2026-01-01") & (data["Quality_d"] == "G") & (data["Quality_s"] == "G") ].drop(columns=["Quality_d","Quality_s","Unnamed: 7","Tidsutsnitt"])
    
    # Convert Current speed and directions to u and v components and calculate depth averaged velocities
    data_da = calculate_depth_averaged_velocities(data)
    data_da.drop_duplicates(subset=["u_depth_averaged","v_depth_averaged"],inplace=True)
    data_da[["u_depth_averaged","v_depth_averaged"]].rename(columns={"u_depth_averaged":"u","v_depth_averaged":"v"}).to_csv(f"../observations/{stations[i]}_u_v_depth_averaged.csv", date_format='%Y-%m-%dT%H:%M:%S')

    # Also save just the u and v at different depths
    data_depths = calculate_velocities(data).dropna()
    data_depths = data_depths[["Measuring depth (m)_d","u","v"]].rename(columns={"Measuring depth (m)_d":"depth"})
    data_depths["z"] = -data_depths["depth"]
    data_depths.drop(columns=["depth"]).to_csv(f"../observations/{stations[i]}_u_v.csv", date_format='%Y-%m-%dT%H:%M:%S')
